In [2]:
# !pip install ultralytics roboflow --quiet

In [1]:
# 2️⃣ Importar librerías
from ultralytics import YOLO
from roboflow import Roboflow
import os
import shutil

In [2]:
def copyDataset(ruta_origen, ruta_destino):
    # Crear carpeta destino si no existe
    if not os.path.exists(ruta_destino):
        os.makedirs(ruta_destino)

    # Copiar archivos sueltos en la raíz
    for archivo in ["data.yaml", "README.dataset.txt", "README.roboflow"]:
        src_file = os.path.join(ruta_origen, archivo)
        dst_file = os.path.join(ruta_destino, archivo)
        if os.path.exists(src_file):
            shutil.copy(src_file, dst_file)

    # Carpeta de splits
    splits = ["train", "test", "valid"]
    for split in splits:
        src_split = os.path.join(ruta_origen, split)
        dst_split = os.path.join(ruta_destino, split)
        os.makedirs(dst_split, exist_ok=True)

        # Copiar carpeta de images tal cual
        images_src = os.path.join(src_split, "images")
        images_dst = os.path.join(dst_split, "images")
        shutil.copytree(images_src, images_dst)

        # Copiar y filtrar labels
        labels_src = os.path.join(src_split, "labels")
        labels_dst = os.path.join(dst_split, "labels")
        os.makedirs(labels_dst, exist_ok=True)

        for label_file in os.listdir(labels_src):
            if label_file.endswith(".txt"):
                src_file = os.path.join(labels_src, label_file)
                dst_file = os.path.join(labels_dst, label_file)
                with open(src_file, "r") as f_in, open(dst_file, "w") as f_out:
                    for line in f_in:
                        if line.startswith("0 "):  # solo clase 0
                            f_out.write(line)

    print("✅ Dataset copiado y filtrado correctamente")


In [3]:
def download_dataset(api_key, workspace, project_name, version=2, output_dir="finetune_dataset_grande"):
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(workspace).project(project_name)
    dataset = project.version(version).download("yolov11")

    # Crear carpeta de salida si no existe
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

        
    dest_path = os.path.join(output_dir, os.path.basename(dataset.location))
    
    if os.path.exists(dest_path):
        shutil.rmtree(dest_path)

    shutil.move(dataset.location, output_dir)

    print(f"✅ Dataset descargado en: {output_dir}")

    copyDataset(output_dir, f"{output_dir}{2}")

In [5]:
def finetune(output_dir="finetune_dataset_grande"):
    # Ruta al data.yaml
    data_yaml = os.path.join(output_dir, "data.yaml")

    # Crear modelo YOLOv11 preentrenado
    model = YOLO("yolo11m.pt")  # cambia a yolov11n.pt para un modelo más ligero

    # Entrenar modelo con tu dataset
    model.train(
        data=data_yaml,
        epochs=50,
        imgsz=640,
        batch=20,
        name="football_finetune_2",
        project="runs/train",
        exist_ok=True
    )

    print("✅ Entrenamiento completado. Pesos guardados en runs/train/football_finetune/weights/")
    return model

In [6]:
API_KEY = "KSNZNjhrcZN1cq1zmo33"
# WORKSPACE = "work-ejvtm"
# PROJECT_NAME = "football-ai-ikmty"
WORKSPACE = "yolo-atnlh"
PROJECT_NAME = "football-detection-wfhdh"

In [7]:
download_dataset(API_KEY, WORKSPACE, PROJECT_NAME, version=2, output_dir="finetune_dataset")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FootBall-Detection-2 in yolov11:: 100%|██████████| 1338/1338 [00:00<00:00, 4769.37it/s]

✅ Dataset descargado en: finetune_dataset


FileNotFoundError: [Errno 2] No such file or directory: 'finetune_dataset/train/images'

In [ ]:
import os
import yaml
import random

# ------------------------------
# CONFIGURACIÓN
# ------------------------------
dataset_dir = "/content/finetune_dataset_grande_filtered"  # carpeta filtrada que tienes
yaml_file = os.path.join(dataset_dir, "data.yaml")

# ------------------------------
# 1️⃣ Mostrar un par de archivos .txt de labels
# ------------------------------
splits = ["train", "valid", "test"]

print("Mostrando un par de labels de cada split...\n")

for split in splits:
    labels_path = os.path.join(dataset_dir, split, "labels")
    if not os.path.exists(labels_path):
        print(f"No se encontró la carpeta: {labels_path}")
        continue
    
    label_files = [f for f in os.listdir(labels_path) if f.endswith(".txt")]
    sample_files = random.sample(label_files, min(2, len(label_files)))  # toma 2 al azar
    
    print(f"--- {split.upper()} ---")
    for f in sample_files:
        print(f"Archivo: {f}")
        with open(os.path.join(labels_path, f), "r") as file:
            lines = file.readlines()
            print("".join(lines[:5]))  # muestra las primeras 5 líneas
    print("\n")

# ------------------------------
# 2️⃣ Modificar el YAML
# ------------------------------
data_yaml = {
    "train": "../train/images",
    "val": "../valid/images",
    "test": "../test/images",
    "nc": 1,
    "names": ["ball"],
    "roboflow": {
        "workspace": "yolo-atnlh",
        "project": "football-detection-wfhdh",
        "version": 2,
        "license": "CC BY 4.0",
        "url": "https://universe.roboflow.com/yolo-atnlh/football-detection-wfhdh/dataset/2"
    }
}

with open(yaml_file, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"✅ YAML modificado en: {yaml_file}")


In [7]:
model = finetune(output_dir="finetune_dataset/football-ai-2")

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=finetune_dataset/football-ai-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=football_finetune_2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar CSV
csv_path = "runs/train/football_finetune_x/results.csv"
df = pd.read_csv(csv_path)

# Losss de entrenamiento y validación
plt.figure(figsize=(10,5))
plt.plot(df['epoch'], df['train/box_loss'], label='train box_loss')
plt.plot(df['epoch'], df['train/cls_loss'], label='train cls_loss')
plt.plot(df['epoch'], df['train/dfl_loss'], label='train dfl_loss')
plt.plot(df['epoch'], df['val/box_loss'], label='val box_loss', linestyle='--')
plt.plot(df['epoch'], df['val/cls_loss'], label='val cls_loss', linestyle='--')
plt.plot(df['epoch'], df['val/dfl_loss'], label='val dfl_loss', linestyle='--')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("YOLOv11 Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# mAP
plt.figure(figsize=(10,5))
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("YOLOv11 Training Accuracy")
plt.legend()
plt.grid(True)
plt.show()
